<a href="https://colab.research.google.com/github/dickyafriza/machine-learning-IMK/blob/main/Klasifikasi_Anomali_IMK_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ====== Install deps ======
!pip -q install scikit-learn pandas numpy joblib chardet

# ====== Import ======
import pandas as pd, numpy as np, re, os, chardet
from io import StringIO
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# ====== 1) Upload CSV (kotor/bersih) ======
print('Upload CSV (boleh kotor/bersih).')
uploaded = files.upload()
raw_name = list(uploaded.keys())[0]
raw_bytes = uploaded[raw_name]
print('Diterima:', raw_name, 'size=', len(raw_bytes))  # [file:52]

# ====== 2) Pembersihan ringan ======
if raw_name.lower().endswith('.xlsx'):
    df = pd.read_excel(raw_bytes)
else:
    enc = (chardet.detect(raw_bytes)['encoding'] or 'utf-8')
    text = raw_bytes.decode(enc, errors='replace')
    text = text.lstrip('\ufeff').replace('\r\n','\n').replace('\r','\n')
    lines = text.split('\n')
    while lines and (lines[0].strip().startswith('**') or lines[0].strip().lower().startswith('mohon') or lines[0].strip().lower().startswith('catatan')):
        lines.pop(0)
    df = pd.read_csv(StringIO('\n'.join(lines)))
df.columns = [str(c).strip() for c in df.columns]
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].astype(str).str.strip()
print('Preview:', df.shape, list(df.columns)[:12])  # [file:52]

# ====== 3) Split Nama Bisnis < Nama Pemilik > dari r213; kosongkan jika <> kosong/placeholder ======
def split_business_owner(series):
    angle_pat = re.compile(r'<([^<>]*)>')  # termasuk kosong
    invalid_tokens = {'', '-', '—', '.', '..', '...'}
    biz, owner_main, owner_others = [], [], []
    for val in series.fillna(''):
        s = str(val).strip()
        s = re.sub(r'\s*<\s*', '<', s)
        s = re.sub(r'\s*>\s*', '>', s)
        raw_owners = angle_pat.findall(s)
        owners = []
        for o in raw_owners:
            oc = re.sub(r'\s+', ' ', o).strip(' <>-_./|')
            if oc.upper() not in invalid_tokens and oc != '':
                owners.append(oc)
        name_raw = angle_pat.sub('', s).strip()
        name_clean = re.sub(r'\s{2,}', ' ', name_raw).strip(' -_/|')
        if not name_clean and '<' in s:
            name_clean = s.split('<', 1)[0].strip()
        biz.append(name_clean)
        owner_main.append(owners[0] if owners else '')
        owner_others.append(', '.join(owners[1:]) if len(owners) > 1 else '')
    return pd.DataFrame({'nama_bisnis': biz,
                         'nama_pemilik': owner_main,
                         'nama_pemilik_lain': owner_others})

if 'r213' in df.columns:
    sp = split_business_owner(df['r213'])
    df = pd.concat([df.drop(columns=['r213']), sp], axis=1)

# ====== 4) Target r216 dua digit (kbli2_true) ======
if 'r216_value' in df.columns:
    df['kbli2_true'] = df['r216_value'].astype(str).str.extract(r'(\d{2})')
elif 'r216_label' in df.columns:
    df['kbli2_true'] = df['r216_label'].astype(str).str.extract(r'\[(\d{2})\]')
else:
    df['kbli2_true'] = np.nan  # [file:52]

# ====== 5) Fitur teks (r215a1_label, r215b, r215d) ======
feat_cols = [c for c in ['r215a1_label','r215b','r215d'] if c in df.columns]
if not feat_cols:
    raise ValueError('Tidak ditemukan kolom r215a1_label/r215b/r215d.')
X_all = df[feat_cols].fillna('')

# ====== 6) Model: latih jika ada y ======
ct = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=5), feat_cols)])
rf = RandomForestClassifier(n_estimators=500, random_state=42, class_weight='balanced_subsample', n_jobs=-1)
pipe = Pipeline([('prep', ct), ('clf', rf)])

has_y = df['kbli2_true'].notna().sum() >= 50 and df['kbli2_true'].nunique() >= 2
if has_y:
    X_t = df.loc[df['kbli2_true'].notna(), feat_cols].fillna('')
    y_t = df.loc[df['kbli2_true'].notna(), 'kbli2_true']
    vc = y_t.value_counts()
    ok = y_t.isin(vc[vc>=2].index)
    if ok.sum() >= 2 and vc[vc>=2].shape[0] >= 2:
        X_tr, X_te, y_tr, y_te = train_test_split(X_t[ok], y_t[ok], test_size=0.2, random_state=42, stratify=y_t[ok])
        pipe.fit(X_tr, y_tr)
        yp = pipe.predict(X_te)
        print('Classification report:')
        print(classification_report(y_te, yp, zero_division=0))
        labels_order = sorted(y_t.unique())
        print(pd.DataFrame(confusion_matrix(y_te, yp, labels=labels_order), index=labels_order, columns=labels_order))
    else:
        pipe.fit(X_t, y_t)
        print('Model dilatih tanpa split (kelas jarang).')
else:
    # fallback fit agar pipeline hidup (tidak untuk mutu evaluasi)
    pipe.fit(X_all, np.random.choice([f'{i:02d}' for i in range(10,34)], size=len(X_all)))

# ====== 7) Prediksi + label nama kategori C ======
pred = pipe.predict(X_all)
proba = pipe.predict_proba(X_all).max(axis=1)

label_map = {
 '10':'Industri Makanan','11':'Industri Minuman','12':'Industri Pengolahan Tembakau','13':'Industri Tekstil',
 '14':'Industri Pakaian Jadi','15':'Industri Kulit dan Alas Kaki','16':'Industri Kayu','17':'Industri Kertas',
 '18':'Industri Pencetakan dan Reproduksi Media Rekaman','19':'Industri Produk dari Batu Bara dan Pengilangan Minyak Bumi',
 '20':'Industri Bahan Kimia dan Barang dari Bahan Kimia','21':'Industri Farmasi, Produk Obat Kimia dan Obat Tradisional',
 '22':'Industri Karet, Barang dari Karet dan Plastik','23':'Industri Barang Galian Bukan Logam','24':'Industri Logam Dasar',
 '25':'Industri Barang dari Logam, Bukan Mesin dan Peralatannya','26':'Industri Komputer, Barang Elektronik dan Optik',
 '27':'Industri Peralatan Listrik','28':'Industri Mesin dan Perlengkapan','29':'Industri Kendaraan Bermotor, Trailer dan Semi Trailer',
 '30':'Industri Alat Angkutan Lainnya','31':'Industri Furnitur','32':'Industri Pengolahan Lainnya',
 '33':'Jasa Reparasi dan Pemasangan Mesin dan Peralatan'
}

out = df.copy()
out['kbli2_pred'] = pred
out['kbli2_pred_label'] = out['kbli2_pred'].map(label_map)
out['kbli2_pred_proba'] = proba

# ====== 8) Aturan iteratif koreksi jelas ======
def apply_iterative_rules_simple(df, cols, max_iters=3, conf_thr=0.70):
    txt = df[cols].fillna('').agg(' '.join, axis=1).str.upper()
    rules = [
        (r'\bKABEL\b|\bTRAFO\b|\bAMPLI(FIER)?\b|\bINVERTER\b', '27'),
        (r'\bCPU\b|\bLAPTOP\b|\bKAMERA\b|\bOPTIK\b', '26'),
        (r'\bMESIN\b|\bDINAMO\b|\bPOMPA\b|\bKOMPRESOR\b', '28'),
        (r'\bKURSI\b|\bMEJA\b|\bLEMARI\b', '31'),
        (r'\bKERTAS\b|\bAGENDA MAP\b', '17'),
        (r'\bCETAK\b|\bPERCETAKAN\b|\bUNDANGAN\b|\bSTIKER\b', '18'),
        (r'\bLEM\b|\bCAT\b|\bRESIN\b', '20'),
        (r'\bKARET\b|\bPLASTIK\b', '22'),
        (r'\bTEPUNG\b|\bSINGKONG\b|\bBERAS\b|\bKUE\b|\bTEMPE\b|\bGETHUK\b|\bTAHU\b', '10'),
        (r'\bAIR MINUM\b|\bSIRUP\b|\bMINUMAN\b', '11'),
    ]
    changed, it = True, 0
    out2 = df.copy()
    while changed and it < max_iters:
        changed, it = False, it+1
        cand = (out2['kbli2_pred_proba'] < conf_thr)
        for pattern, target in rules:
            m = cand & txt.str.contains(pattern, regex=True, na=False) & (out2['kbli2_pred'] != target)
            if m.any():
                out2.loc[m, 'kbli2_pred'] = target
                out2.loc[m, 'kbli2_pred_label'] = out2.loc[m, 'kbli2_pred'].map(label_map)
                changed = True
    return out2

out_iter = apply_iterative_rules_simple(out, feat_cols, max_iters=3, conf_thr=0.70)

# ====== 9) Kategori C & status kesesuaian dengan r216 ======
catC = [f"{i:02d}" for i in range(10,34)]
out_iter['is_catC_pred'] = out_iter['kbli2_pred'].isin(catC)
out_iter['is_catC_true'] = out_iter['kbli2_true'].isin(catC)
mismatch = out_iter['kbli2_true'].notna() & (out_iter['kbli2_true'] != out_iter['kbli2_pred'])
out_iter['status_kesesuaian'] = np.where(
    out_iter['is_catC_pred'] & out_iter['is_catC_true'] & (~mismatch), 'Sesuai C',
    np.where(~out_iter['is_catC_pred'] & out_iter['is_catC_true'], 'True C vs Pred non-C',
             np.where(out_iter['is_catC_pred'] & ~out_iter['is_catC_true'], 'True non-C vs Pred C', 'True non-C & Pred non-C'))
)

# ====== 10) Bagi output ======
klasifikasi = out_iter.copy()
bersih = out_iter.loc[out_iter['is_catC_pred'] & out_iter['is_catC_true'] & (~mismatch)].copy()
anomali = out_iter.loc[(~out_iter['is_catC_pred']) | (~out_iter['is_catC_true']) | mismatch].copy()

# Pastikan kolom utama tetap ada
for dfx in [klasifikasi, bersih, anomali]:
    for col in ['r215a1_label','r215b','r215d','r216_label']:
        if col not in dfx.columns and col in df.columns:
            dfx[col] = df[col]

# ====== 11) Simpan & download (3 file) ======
stamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
klasifikasi_name = f'klasifikasi_r216_vs_textC_{stamp}.csv'
bersih_name      = f'bersih_textC_{stamp}.csv'
anom_name        = f'anomali_kbli_{stamp}.csv'

id_cols = [c for c in ['r101','r102','r103','r104','r105','r106','r107','r206','r208'] if c in klasifikasi.columns]
show_cols = id_cols + [c for c in ['nama_bisnis','nama_pemilik','r215a1_label','r215b','r215d','r216_label','kbli2_true','kbli2_pred','kbli2_pred_label','kbli2_pred_proba','status_kesesuaian'] if c in klasifikasi.columns]
klasifikasi[show_cols].to_csv(klasifikasi_name, index=False)

# Bersih: hanya label final yang bersih + kolom diminta
bersih_cols = id_cols + [c for c in ['nama_bisnis','nama_pemilik','r215a1_label','r215b','r215d','r216_label','kbli2_pred','kbli2_pred_label'] if c in bersih.columns]
bersih[bersih_cols].to_csv(bersih_name, index=False)

anomali[show_cols].to_csv(anom_name, index=False)

print('Saved:', klasifikasi_name, 'size=', os.path.getsize(klasifikasi_name))
print('Saved:', bersih_name,      'size=', os.path.getsize(bersih_name))
print('Saved:', anom_name,        'size=', os.path.getsize(anom_name))

# Auto-download ketiga file
for fn in [klasifikasi_name, bersih_name, anom_name]:
    if os.path.exists(fn):
        files.download(fn)
        print('Mengunduh:', fn, 'size=', os.path.getsize(fn))
    else:
        print('File tidak ditemukan:', fn)

# ====== 12) Simpan model (opsional) ======
joblib.dump(pipe, 'model_kbli2_rf.joblib')

Upload CSV (boleh kotor/bersih).


### Distribusi Kode KBLI yang Diprediksi

# saranku (Ivan)

- coba kurangin max_depth nya
